# Isoform Enumeration Inspection

This notebook inspects the isoform enumeration pipeline (tautomers, protonation states, neutralization).
It runs enumeration on diverse test molecules, highlights known edge cases, and validates all output SMILES.

In [ ]:
import pandas as pd
from rdkit import Chem, RDLogger

from golem.config import IsoformConfig
from golem.isoforms import (
    _enumerate_protonation,
    _enumerate_tautomers,
    _neutralize,
    enumerate_isoforms,
)

# Suppress RDKit warnings in notebook output
RDLogger.DisableLog('rdApp.*')

## 1. Test Molecules

A diverse set covering acids, bases, amides, heterocycles, and drug-like molecules.

In [ ]:
test_molecules = {
    "Acetic acid": "CC(=O)O",
    "Ethylamine": "CCN",
    "Glycine": "NCC(=O)O",
    "N-acetylpiperidine (tertiary amide)": "CC(=O)N1CCCCC1",
    "Indole": "c1ccc2[nH]ccc2c1",
    "Pyrrole": "c1cc[nH]c1",
    "Guanine (tautomers)": "O=c1[nH]c2[nH]cnc2c(=O)[nH]1",
    "Phenol": "Oc1ccccc1",
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen": "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
}

print(f"Testing {len(test_molecules)} molecules")

## 2. Run Isoform Enumeration

In [ ]:
config = IsoformConfig()
print(f"Config: max_tautomers={config.max_tautomers}, max_protomers={config.max_protomers}, ph_range={config.ph_range}")

results = []
for name, smi in test_molecules.items():
    isoforms = enumerate_isoforms(smi, config)
    results.append({
        "name": name,
        "parent_smiles": smi,
        "n_isoforms": len(isoforms),
        "isoforms": isoforms,
    })

df = pd.DataFrame(results)
df[["name", "parent_smiles", "n_isoforms"]]

## 3. Detailed Isoform Listing

In [ ]:
for _, row in df.iterrows():
    print(f"\n{'='*60}")
    print(f"{row['name']}  ({row['parent_smiles']})")
    print(f"  {row['n_isoforms']} isoforms:")
    for i, iso in enumerate(row['isoforms']):
        tag = " (original)" if i == 0 else ""
        print(f"    [{i}] {iso}{tag}")

## 4. Edge Case Inspection

### 4a. Tertiary Amide — should NOT be protonated

In [ ]:
amide_smi = "CC(=O)N1CCCCC1"
protomers = _enumerate_protonation(amide_smi, ph_range=(6.4, 8.4), max_protomers=10)
print(f"Tertiary amide protomers ({len(protomers)}):")
for pmol in protomers:
    psmi = Chem.MolToSmiles(pmol, canonical=True)
    has_charged_n = any(
        a.GetAtomicNum() == 7 and a.GetFormalCharge() > 0 for a in pmol.GetAtoms()
    )
    flag = " *** PROTONATED N (bad) ***" if has_charged_n else " (ok)"
    print(f"  {psmi}{flag}")

### 4b. Nitrogen with 4+ Hydrogens — should be filtered

In [ ]:
# Check that no protomer in our results has N with >= 4 H
n4h_count = 0
for _, row in df.iterrows():
    for iso in row['isoforms']:
        mol = Chem.MolFromSmiles(iso)
        if mol is None:
            continue
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 7 and atom.GetTotalNumHs() >= 4:
                print(f"  N(4+H) found in: {iso} (parent: {row['parent_smiles']})")
                n4h_count += 1

if n4h_count == 0:
    print("No N(4+H) found in any isoform output.")
else:
    print(f"\nWARNING: {n4h_count} N(4+H) instances found!")

## 5. Validate All Output SMILES

In [ ]:
total = 0
valid = 0
invalid = []

for _, row in df.iterrows():
    for iso in row['isoforms']:
        total += 1
        mol = Chem.MolFromSmiles(iso)
        if mol is not None:
            try:
                Chem.SanitizeMol(mol)
                valid += 1
            except Exception:
                invalid.append((row['name'], iso))
        else:
            invalid.append((row['name'], iso))

print(f"Total isoforms: {total}")
print(f"Valid: {valid}")
print(f"Invalid: {len(invalid)}")
if invalid:
    print("\nInvalid SMILES:")
    for name, smi in invalid:
        print(f"  {name}: {smi}")

## 6. Summary Statistics

In [ ]:
total_parents = len(df)
total_isoforms = df['n_isoforms'].sum()
mean_expansion = df['n_isoforms'].mean()
max_expansion = df.loc[df['n_isoforms'].idxmax()]

print(f"Parents: {total_parents}")
print(f"Total isoforms: {total_isoforms}")
print(f"Mean expansion: {mean_expansion:.1f}x")
print(f"Max expansion: {max_expansion['n_isoforms']}x ({max_expansion['name']})")
print(f"Failures (invalid SMILES): {len(invalid)}")
print(f"\nExpansion per molecule:")
for _, row in df.iterrows():
    bar = '#' * row['n_isoforms']
    print(f"  {row['name']:40s} {row['n_isoforms']:3d}  {bar}")